# RAPID feature-timing comparison animation

`TIMING_FEATURE = 'centroid'` moves markers using the local excess-volume centroid; use `'peak'` to restore peak timing. Timing is computed on the full reviewed response window, independently of the movie crop. The centroid is a time coordinate, not a separate discharge peak.

For local map hydrographs, the reviewed baseline is scaled by the link's reference flow fraction and lowered to the local window minimum only if needed to keep excess nonnegative. At the outlet, the shared reviewed baseline and window reproduce the hydrograph-response centroid. One-cell parent links use their outlet observation as a midpoint proxy.

The combined view shows static diamond markers and dotted lines at the outlet centroids. The moving cursor still represents shared simulation time, meeting each diamond when that outlet's centroid time is reached. Its moving discharge markers are instantaneous discharge, not a moving centroid or peak height.

Restart the kernel and Run All. Use the output switches to skip optional GIFs; run the setup, data-loading and timing-calculation cells first. Each export initializes its own drawing helpers. The full base geometry is shown with variant geometry inside its validated collapse domain. Both trajectories are retained.


In [ ]:
network = 37
TARGET_STATE = 'CB001_bubble_3'  # change to any state in rapid_run_registry.csv


In [16]:
from pathlib import Path
import sys

# Locate the checkout from either the repository root or a notebook subdirectory.
REPO = next((path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (path / 'RAPID' / 'src').is_dir()
             and (path / 'network_variants').is_dir()), None)
if REPO is None:
    raise RuntimeError('Open this notebook from inside the river-hierarchy checkout.')
# Cunge also imports the sibling network_variants package.
for import_root in (REPO, REPO / 'RAPID' / 'src'):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

EXPERIMENT = REPO / f'network_variants/output/{network}_compound_bubbles__anchored'
REFERENCE_STATE = 'S000_base'
RESPONSE_WINDOW = '0.5%'  # same shared boundaries and reruns as hydrograph_response
OUTPUT_GIF = EXPERIMENT / f'{TARGET_STATE}_vs_{REFERENCE_STATE}_peak_dots.gif'

# Peak definition. Midpoint is recommended when peaks occupy several time steps.
TIMING_FEATURE = 'centroid'  # 'centroid' or 'peak'
PEAK_RULE = 'midpoint_of_plateau'  # 'first_max' or 'midpoint_of_plateau'
PEAK_RELATIVE_TOLERANCE = 1e-6
WAVE_WIDTH_RELATIVE_LEVEL = 0.80  # shoulder dots: rising/receding fraction of local peak excess; use 0.50 for broader pulses
OBSERVED_SPEED_REGULARIZATION = 0.03  # weak anchor to RAPID K when peak equations are underdetermined

# Motion: 'midpoint_timing' (original), 'hybrid_midpoint_timing',
# 'observed_link_speed', 'network_observed_link_speed', or 'celerity'. The hybrid keeps exact peak times
# but smooths speed changes at shared link midpoints.
MOTION_MODE = 'hybrid_midpoint_timing'
# Playback only: hours relative to the reviewed response-window start.
# Use None for either bound to retain that end of the full response window.
# None uses the full response window; centroid passage occurs later than peak passage.
MOVIE_START_HOUR = None
MOVIE_END_HOUR = None
PLAYBACK_SECONDS = 20.0  # lower = faster (15); higher = slower (60)
N_FRAMES = 360
HYBRID_N_FRAMES = 720  # spatial/time smoothness; speed is controlled above


from rapid_tools import animation_notebook
import importlib
importlib.reload(animation_notebook)
animation_notebook.prepare_playback(globals())


DOT_SIZE = 38

# Optional outputs: keep calculation/helper cells enabled and use these switches.
SHOW_PREVIEW = False
RENDER_SIDE_BY_SIDE = False
RENDER_OVERLAY = False

SCALE_BAR_KM = None  # automatic 1/2/5-based length; set e.g. 10 for a 10 km bar


In [ ]:
import numpy as np
import pandas as pd
# Refresh local source when rerunning an already-open notebook after code edits.
import importlib
from rapid_tools import cunge, cunge_wave
importlib.reload(cunge)
importlib.reload(cunge_wave)
from rapid_tools.cunge_wave import load_cunge_wave_state

registry = pd.read_csv(EXPERIMENT / 'rapid_run_registry.csv')
available_states = registry.loc[registry.status.eq('ran'), 'state_id'].tolist()
print(f'{len(available_states)} completed states (not necessarily animation-ready): {available_states}')
assert REFERENCE_STATE in available_states
assert TARGET_STATE in available_states

reference = load_cunge_wave_state(EXPERIMENT, REFERENCE_STATE, reviewed_window=RESPONSE_WINDOW)
target    = load_cunge_wave_state(EXPERIMENT, TARGET_STATE, reviewed_window=RESPONSE_WINDOW)
def common_elapsed_time(reference, target):
    """Align RAPID outputs by elapsed time since their own event starts."""
    if pd.Timestamp(reference.run_row.event_start_time_utc) != pd.Timestamp(target.run_row.event_start_time_utc):
        raise ValueError('Base and variant forcing origins differ; cannot share an elapsed-time clock.')
    reference_time = reference.wave_stack.time_seconds - float(reference.run_row['event_start_time_seconds'])
    target_time = target.wave_stack.time_seconds - float(target.run_row['event_start_time_seconds'])
    time, reference_index, target_index = np.intersect1d(
        reference_time, target_time, return_indices=True
    )
    # Do not animate pre-event values, and retain only the shared simulation extent.
    keep = time >= 0
    time = time[keep]
    reference_index, target_index = reference_index[keep], target_index[keep]
    if time.size == 0:
        raise ValueError('The two runs have no common post-event RAPID timesteps.')
    return time, reference.wave_stack.qout[reference_index], target.wave_stack.qout[target_index]

time_seconds, reference_qout, target_qout = common_elapsed_time(reference, target)
print(f'Common post-event window: {len(time_seconds)} steps, {time_seconds[-1] / 3600:.1f} hours')
print('Both event starts have UTC timestamp:', reference.run_row.event_start_time_utc, target.run_row.event_start_time_utc)

origin = pd.Timestamp(reference.run_row.event_start_time_utc)
ANIMATION_START_SECONDS = (pd.Timestamp(reference.run_row.animation_start_utc) - origin).total_seconds()
ANIMATION_END_SECONDS = (pd.Timestamp(reference.run_row.animation_end_utc) - origin).total_seconds()
assert reference.run_row.animation_start_utc == target.run_row.animation_start_utc
assert reference.run_row.animation_end_utc == target.run_row.animation_end_utc
INLET_FEATURE_SECONDS = float(reference.run_row.forcing_peak_time_seconds)
print(f'Movie window ({RESPONSE_WINDOW}): {reference.run_row.animation_start_utc} to {reference.run_row.animation_end_utc}')
print('Full routing data retained for peak extraction; only the movie clock is cropped.')

display(pd.DataFrame([reference.run_row, target.run_row])[
    ['state_id', 'wave_source', 'outlet_parquet', 'forcing_parquet', 'animation_start_utc', 'animation_end_utc']])

from rapid_tools.cunge_wave import wave_centroid
if TIMING_FEATURE not in {'peak', 'centroid'}:
    raise ValueError("TIMING_FEATURE must be 'peak' or 'centroid'.")
forcing_for_timing = pd.read_parquet(reference.run_row.forcing_parquet)
forcing_seconds = (pd.DatetimeIndex(forcing_for_timing.time_utc) - origin).total_seconds().to_numpy()
if TIMING_FEATURE == 'centroid':
    INLET_FEATURE_SECONDS, _ = wave_centroid(
        forcing_seconds, forcing_for_timing.forcing_cms.to_numpy(),
        ANIMATION_START_SECONDS, ANIMATION_END_SECONDS, reference.run_row.centroid_baseline_cms)


In [ ]:
import importlib
from rapid_tools import cunge_wave
importlib.reload(cunge_wave)
from rapid_tools.cunge_wave import wave_centroid

def parent_peak_frame(reaches, qout, time_seconds, *, peak_rule, relative_tolerance, baseline_cms):
    """Use local midpoint discharge for peak or centroid timing; one-cell links use an outlet proxy."""
    frame = reaches[['parent_link_id', 'id_us_node', 'id_ds_node', 'subreach_index', 'rapid_k', 'reference_flow_fraction', 'geometry']].copy()
    frame['parent_link_id'] = pd.to_numeric(frame['parent_link_id'], errors='coerce')
    frame = frame.loc[frame.parent_link_id.notna()].copy()
    frame['parent_link_id'] = frame.parent_link_id.astype(int)
    # RAPID splits each parent link into subreaches. Preserve the parent endpoints:
    # first subreach upstream node and *last* subreach downstream node.
    endpoint_metadata = (
        frame.sort_values(['parent_link_id', 'subreach_index'])
        .groupby('parent_link_id', as_index=False)
        .agg(id_us_node=('id_us_node', 'first'), id_ds_node=('id_ds_node', 'last'))
    )
    parent_geometry = frame[['parent_link_id', 'geometry']].dissolve(by='parent_link_id', as_index=False)
    # RAPID stores each parent link as several subreaches.  Merge contiguous
    # pieces so line styles (especially dashes) run along the whole link.
    from shapely.ops import linemerge
    parent_geometry['geometry'] = parent_geometry.geometry.map(
        lambda geometry: linemerge(geometry) if geometry.geom_type == 'MultiLineString' else geometry
    )
    parent_geometry = parent_geometry.merge(endpoint_metadata, on='parent_link_id', validate='one_to_one')
    parent_ids = frame.parent_link_id.to_numpy()
    rows = []
    timestep_seconds = float(np.median(np.diff(time_seconds)))

    def peak_index(series):
        peak = np.nanmax(series)
        if peak_rule == 'first_max':
            return int(np.nanargmax(series))
        if peak_rule == 'midpoint_of_plateau':
            indices = np.flatnonzero(np.isclose(series, peak, rtol=relative_tolerance, atol=0.0))
            return int(np.rint(np.mean(indices)))
        raise ValueError(f'Unknown peak rule: {peak_rule}')

    for parent_link_id in parent_geometry.parent_link_id:
        parent = frame.loc[frame.parent_link_id.eq(parent_link_id)].sort_values('subreach_index')
        parent_mask = parent_ids == parent_link_id
        # Each saved subreach discharge is at its downstream endpoint.
        # Interpolate the two observations bracketing the parent midpoint;
        # averaging the entire link can select a different hydrograph crest.
        from rapid_tools.cunge_wave import midpoint_discharge
        series = midpoint_discharge(qout[:, parent.index.to_numpy()])
        peak_at = peak_index(series)
        local_baseline = float(baseline_cms) * float(parent.reference_flow_fraction.iloc[0])
        centroid_time, centroid_baseline = wave_centroid(
            time_seconds, series, ANIMATION_START_SECONDS, ANIMATION_END_SECONDS, local_baseline)
        def local_time(values):
            if TIMING_FEATURE == 'peak':
                return time_seconds[peak_index(values)]
            return wave_centroid(time_seconds, values, ANIMATION_START_SECONDS,
                                 ANIMATION_END_SECONDS, local_baseline)[0]
        feature_time = time_seconds[peak_at] if TIMING_FEATURE == 'peak' else centroid_time
        # Width of the local hydrograph, measured at a configurable fraction of
        # its excess above the pre-event/local minimum. This controls pulse length.
        baseline, peak = np.nanmin(series), np.nanmax(series)
        threshold = baseline + WAVE_WIDTH_RELATIVE_LEVEL * (peak - baseline)
        above_threshold = series >= threshold
        # Use only the connected threshold interval containing this selected
        # peak, rather than unrelated high-flow periods elsewhere in the run.
        left = right = peak_at
        while left > 0 and above_threshold[left - 1]:
            left -= 1
        while right < len(series) - 1 and above_threshold[right + 1]:
            right += 1
        wave_width = time_seconds[right] - time_seconds[left]
        # Observed within-link travel time uses first and last RAPID subreaches.
        # A one-subreach parent has no internal observation, so falls back to K.
        configured_seconds = float(pd.to_numeric(parent.rapid_k, errors='coerce').sum())
        observed_seconds = configured_seconds
        if len(parent) > 1:
            upstream = np.nanmean(qout[:, parent_mask & frame.subreach_index.eq(parent.subreach_index.iloc[0]).to_numpy()], axis=1)
            downstream = np.nanmean(qout[:, parent_mask & frame.subreach_index.eq(parent.subreach_index.iloc[-1]).to_numpy()], axis=1)
            observed_seconds = local_time(downstream) - local_time(upstream)
        observed_seconds = float(observed_seconds) if np.isfinite(observed_seconds) and observed_seconds > 0 else configured_seconds
        rows.append({'parent_link_id': parent_link_id, 'timing_time_seconds': feature_time, 'peak_time_seconds': time_seconds[peak_at],
                     'centroid_time_seconds': centroid_time, 'centroid_baseline_cms': centroid_baseline,
                     'peak_width_seconds': max(float(wave_width), timestep_seconds),
                     'observed_link_travel_seconds': max(float(observed_seconds), timestep_seconds),
                     'celerity_link_travel_seconds': max(configured_seconds, timestep_seconds)})
    peaks = pd.DataFrame(rows)
    result = parent_geometry.merge(peaks, on='parent_link_id', validate='one_to_one')
    points = result.geometry.interpolate(0.5, normalized=True)
    result['x'] = points.x
    result['y'] = points.y
    return result

base_peaks = parent_peak_frame(reference.wave_stack.reaches, reference_qout, time_seconds, peak_rule=PEAK_RULE, relative_tolerance=PEAK_RELATIVE_TOLERANCE, baseline_cms=reference.run_row.centroid_baseline_cms)
variant_peaks = parent_peak_frame(target.wave_stack.reaches, target_qout, time_seconds, peak_rule=PEAK_RULE, relative_tolerance=PEAK_RELATIVE_TOLERANCE, baseline_cms=target.run_row.centroid_baseline_cms)
matched = base_peaks[['parent_link_id', 'timing_time_seconds']].merge(
    variant_peaks[['parent_link_id', 'timing_time_seconds']], on='parent_link_id', suffixes=('_base', '_variant')
)
matched['delta_timing_hours'] = (matched.timing_time_seconds_variant - matched.timing_time_seconds_base) / 3600
display(matched.delta_timing_hours.describe())

def subreach_wave_field(reaches, qout):
    """Local RAPID hydrographs for a spatial, non-icon wave rendering."""
    paths = []
    columns = []
    for column, geometry in enumerate(reaches.geometry):
        parts = geometry.geoms if geometry.geom_type == 'MultiLineString' else [geometry]
        for part in parts:
            paths.append(np.asarray(part.coords))
            columns.append(column)
    baseline = np.nanmin(qout, axis=0)
    scale = np.maximum(np.nanmax(qout, axis=0) - baseline, np.finfo(float).eps)
    return {'paths': paths, 'columns': np.asarray(columns), 'qout': qout, 'baseline': baseline, 'scale': scale}

base_wave_field = subreach_wave_field(reference.wave_stack.reaches, reference_qout)
variant_wave_field = subreach_wave_field(target.wave_stack.reaches, target_qout)

# Overlay map: show the variant only inside its validated collapse domain.
# Full peak tables remain available for timing and marker trajectories.
from rapid_tools.cunge_wave import variant_domain_geometry
variant_overlay_peaks, overlay_domain = variant_domain_geometry(reference, target, variant_peaks)
variant_domain_mask = target.wave_stack.reaches.parent_link_id.isin(overlay_domain['child_links']).to_numpy()
variant_overlay_wave_field = subreach_wave_field(
    target.wave_stack.reaches.loc[variant_domain_mask], target_qout[:, variant_domain_mask])
print(f"Variant overlay: node {overlay_domain['child_source']} → {overlay_domain['child_sink']}; "
      f"{len(variant_overlay_peaks)} links. Full base geometry retained.")


In [ ]:
import matplotlib.pyplot as plt

def draw_network(ax, links, active, *, title, bounds):
    links.plot(ax=ax, color='#9aa0a6', linewidth=1.0, zorder=1)
    ax.scatter(active.x, active.y, s=DOT_SIZE, c='#d62728', edgecolors='white', linewidths=0.45, zorder=2)
    ax.set(xlim=(bounds[0], bounds[2]), ylim=(bounds[1], bounds[3]), title=title)
    ax.set_aspect('equal')
    ax.set_axis_off()

all_bounds = np.array([
    min(base_peaks.total_bounds[0], variant_peaks.total_bounds[0]),
    min(base_peaks.total_bounds[1], variant_peaks.total_bounds[1]),
    max(base_peaks.total_bounds[2], variant_peaks.total_bounds[2]),
    max(base_peaks.total_bounds[3], variant_peaks.total_bounds[3]),
])
if SHOW_PREVIEW:
    preview_time = np.median(time_seconds)
    time_step = np.median(np.diff(time_seconds))
    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    draw_network(axes[0], base_peaks, base_peaks[np.abs(base_peaks.timing_time_seconds - preview_time) <= time_step / 2], title=REFERENCE_STATE, bounds=all_bounds)
    draw_network(axes[1], variant_peaks, variant_peaks[np.abs(variant_peaks.timing_time_seconds - preview_time) <= time_step / 2], title=TARGET_STATE, bounds=all_bounds)
    fig.suptitle(f'Peak-dot preview at {(preview_time - time_seconds[0]) / 3600:.1f} h after event-window start')
    plt.show()


In [ ]:
# Initialize drawing helpers. Each export cell also does this automatically.
from rapid_tools.animation_notebook import prepare_animation
prepare_animation(globals())


## Drawing helpers

Drawing helpers live in `rapid_tools/_animation_notebook_helpers.py`. Each GIF-export cell loads them automatically. After running the network selection, setup, data-loading and timing-calculation cells, you can go straight to the combined GIF cell. Optional preview, appearance, and standalone export cells may be skipped.


In [ ]:
# Optional helper refresh; safe to skip when going directly to an export cell.
from rapid_tools.animation_notebook import prepare_animation
prepare_animation(globals())


## Base vs variant animation

This view puts the variant next to the base network. 


Adjust `MOVIE_START_HOUR`, `MOVIE_END_HOUR`, and `PLAYBACK_SECONDS` in the setup cell. Hours use the **response-window clock shown in the GIF**, not GIF frame numbers. Use `None` / `None` for the full reviewed window. After the data have been loaded, rerun the setup cell and your chosen GIF-render cell; reconstruction need not run again if only playback settings change. Titles retain the original hour labels after cropping.


In [ ]:
from rapid_tools.animation_notebook import prepare_animation
prepare_animation(globals())

if RENDER_SIDE_BY_SIDE:
    frame_times, frame_duration_ms = movie_playback(MOTION_MODE)
    with TemporaryDirectory(prefix='rapid-peak-dot-') as tmp_name:
        tmp_dir = Path(tmp_name)
        frames = []
        for index, current_time in enumerate(frame_times):
            fig, axes = plt.subplots(1, 2, figsize=(13, 6))
            draw_movers(axes[0], base_peaks, base_movers, current_time, title=REFERENCE_STATE)
            draw_movers(axes[1], variant_peaks, variant_movers, current_time, title=TARGET_STATE)
            elapsed_hours = (current_time - ANIMATION_START_SECONDS) / 3600
            fig.suptitle(f'{TIMING_FEATURE} timing: {TARGET_STATE} vs {REFERENCE_STATE} | window +{elapsed_hours:.1f} h')
            frame_path = tmp_dir / f'frame_{index:04d}.png'
            fig.savefig(frame_path, dpi=130, bbox_inches='tight')
            plt.close(fig)
            with Image.open(frame_path) as image:
                # Keep RGB frames. Pillow quantizes them together while writing the GIF;
                # pre-quantizing each frame can create incompatible palettes in Pillow 3.13.
                frames.append(image.convert('RGB').copy())

    OUTPUT_GIF.parent.mkdir(parents=True, exist_ok=True)
    frames[0].save(OUTPUT_GIF, save_all=True, append_images=frames[1:], duration=frame_duration_ms, loop=0, disposal=2, optimize=False)
    print(f'Wrote {OUTPUT_GIF} ({len(frames)} frames)')
else:
    print('Side-by-side GIF skipped (RENDER_SIDE_BY_SIDE=False).')


## Overlaid base–variant animation

The full base network is shown together with the variant geometry **only between its validated collapse boundaries**. Boundaries and link membership are looked up for the selected network/state, including aliases; no node IDs are hard-coded. The `zorder` values in `draw_overlay` control which network is on top.

Both marker trajectories and their timing remain unchanged. The side-by-side comparison still shows both complete networks.


In [ ]:
# Optional appearance settings. Defaults also work if this cell is skipped.
BASE_MOVER_STYLE = 'dot'
VARIANT_MOVER_STYLE = 'dot'
WAVE_STYLE = 'dot'
PULSE_MIN_LINK_FRACTION = 0.05
PULSE_MAX_LINK_FRACTION = 0.34
PULSE_LINEWIDTH = 6.0
PULSE_ALPHA = 0.28
PULSE_OVERLAP_TOLERANCE_M = 250.0
SHARED_PULSE_COLOR = '#7c8790'
VARIANT_DASH_PATTERN = (0, (4, 2))
KART_SPRITE_ZOOM = 0.020


In [ ]:
from rapid_tools.animation_notebook import prepare_animation
prepare_animation(globals())

# `dot` is the original/normal peak-marker view. The two pulse styles use
# the locally measured hydrograph width calculated above.
MOTION_MODES_TO_SAVE = ['hybrid_midpoint_timing']
WAVE_STYLES_TO_SAVE = ['dot']  # single markers; use hydro_pulse_dots for shoulders


def draw_overlay(ax, current_time):
    if WAVE_STYLE == 'reference_peak_wave':
        # Reference-only: blue base geometry and a red travelling peak front.
        base_peaks.plot(ax=ax, color='#1976d2', linewidth=1.45, alpha=0.78, zorder=1)
        draw_mover_layer(ax, base_movers, current_time, color='#d62728', style='dot', sprite=None, zoom=0, marker='o', dot_size=DOT_SIZE, zorder=3, draw_pulse=False)
        ax.set(xlim=(all_bounds[0], all_bounds[2]), ylim=(all_bounds[1], all_bounds[3]), aspect='equal')
        ax.set_axis_off()
        draw_scale_bar(ax)
        return
    # Keep the full base; draw variant geometry only within the selected collapse boundaries.
    variant_overlay_peaks.plot(ax=ax, color='#474D4F', linewidth=1.3, linestyle=VARIANT_DASH_PATTERN, alpha=0.95, zorder=2)
    base_peaks.plot(ax=ax, color='#B2BBC2', linewidth=1.35, zorder=1)
    if WAVE_STYLE == 'spatial_wave_field':
        # This is not a travelling icon: it is RAPID's local discharge state
        # along every subreach, so width and intensity remain physically local.
        draw_spatial_wave_field(ax, variant_overlay_wave_field, current_time, color='#1976d2', zorder=5)
        draw_spatial_wave_field(ax, base_wave_field, current_time, color='#d62728', zorder=3)
    elif WAVE_STYLE in {'peak_plus_width', 'hydro_pulse'}:
        variant_pulse = pulse_geometry(variant_movers, current_time)
        base_pulse = pulse_geometry(base_movers, current_time)
        if base_pulse is not None and variant_pulse is not None:
            # Exact line equality is too strict: tiny timing/width differences
            # create distracting coloured fragments. Use the base centreline as
            # the neutral shared front wherever the variant is nearby.
            base_near_variant = base_pulse.intersection(variant_pulse.buffer(PULSE_OVERLAP_TOLERANCE_M))
            base_only = base_pulse.difference(variant_pulse.buffer(PULSE_OVERLAP_TOLERANCE_M))
            variant_only = variant_pulse.difference(base_pulse.buffer(PULSE_OVERLAP_TOLERANCE_M))
        else:
            base_near_variant, base_only, variant_only = None, base_pulse, variant_pulse
        draw_pulse_geometry(ax, variant_only, color='#1976d2', zorder=3)
        draw_pulse_geometry(ax, base_near_variant, color=SHARED_PULSE_COLOR, zorder=4)
        draw_pulse_geometry(ax, base_only, color='#d62728', zorder=5)
        if WAVE_STYLE == 'peak_plus_width':
            draw_mover_layer(ax, variant_movers, current_time, color='#1976d2', style=VARIANT_MOVER_STYLE, sprite=VARIANT_KART_SPRITE, zoom=KART_SPRITE_ZOOM, marker='^', dot_size=DOT_SIZE * 1.55, zorder=6, draw_pulse=False)
            draw_mover_layer(ax, base_movers, current_time, color='#d62728', style=BASE_MOVER_STYLE, sprite=BASE_KART_SPRITE, zoom=KART_SPRITE_ZOOM, marker='o', dot_size=DOT_SIZE, zorder=7, draw_pulse=False)
    else:
        draw_mover_layer(ax, variant_movers, current_time, color='#1976d2', style=VARIANT_MOVER_STYLE, sprite=VARIANT_KART_SPRITE, zoom=KART_SPRITE_ZOOM, marker='^', dot_size=DOT_SIZE * 1.55, zorder=3)
        draw_mover_layer(ax, base_movers, current_time, color='#d62728', style=BASE_MOVER_STYLE, sprite=BASE_KART_SPRITE, zoom=KART_SPRITE_ZOOM, marker='o', dot_size=DOT_SIZE, zorder=5)
    ax.set(xlim=(all_bounds[0], all_bounds[2]), ylim=(all_bounds[1], all_bounds[3]), aspect='equal')
    ax.set_axis_off()
    draw_scale_bar(ax)

# written_gifs = []
# if RENDER_OVERLAY:
#     for MOTION_MODE in MOTION_MODES_TO_SAVE:
#         base_movers = build_movers(base_peaks, MOTION_MODE)
#         variant_movers = build_movers(variant_peaks, MOTION_MODE)
#         frame_times, frame_duration_ms = movie_playback(MOTION_MODE)
#         for WAVE_STYLE in WAVE_STYLES_TO_SAVE:
#             overlay_gif = EXPERIMENT / f'{TARGET_STATE}_vs_{REFERENCE_STATE}_{TIMING_FEATURE}_dots_overlay_{MOTION_MODE}_{WAVE_STYLE}.gif'
#             with TemporaryDirectory(prefix='rapid-peak-dot-overlay-') as tmp_name:
#                 tmp_dir = Path(tmp_name)
#                 frames = []
#                 for index, current_time in enumerate(frame_times):
#                     fig, ax = plt.subplots(figsize=(8.5, 7.5))
#                     draw_overlay(ax, current_time)
#                     elapsed_hours = (current_time - ANIMATION_START_SECONDS) / 3600
#                     fig.suptitle(f'{TIMING_FEATURE} / {MOTION_MODE} / {WAVE_STYLE}: {TARGET_STATE} vs {REFERENCE_STATE} | window +{elapsed_hours:.1f} h')
#                     frame_path = tmp_dir / f'frame_{index:04d}.png'
#                     fig.savefig(frame_path, dpi=140, bbox_inches='tight')
#                     plt.close(fig)
#                     with Image.open(frame_path) as image:
#                         # Keep RGB; per-frame palette conversion can give Pillow
#                         # incompatible palettes when it writes an animated GIF.
#                         frames.append(image.convert('RGB').copy())
#             frames[0].save(overlay_gif, save_all=True, append_images=frames[1:], duration=frame_duration_ms, loop=0, disposal=2, optimize=False)
#             written_gifs.append(overlay_gif)
#             print(f'Wrote {overlay_gif.name} ({len(frame_times)} frames)')
#             del frames

#     print(f'Wrote {len(written_gifs)} overlay GIFs.')
# else:
#     print('Overlay GIF skipped (RENDER_OVERLAY=False).')


## Synchronized map and outlet hydrographs

The map and moving hydrograph cursor use the **same frame timestamps and GIF delays**. Red is the base; blue is the variant. The hydrographs show the saved outlet discharge, with markers interpolated at each animation time. Map dots indicate the selected timing feature (centroid by default); they are not water particles. Dotted lines and diamonds show the fixed outlet centroids; the moving cursor is shared simulation time.

Set `HYDROGRAPH_VIEW = 'movie'` for the same horizontal time range as the cropped movie, or `'response'` to show the complete reviewed response window. Use the existing `MOVIE_START_HOUR`, `MOVIE_END_HOUR`, and `PLAYBACK_SECONDS` controls. Set `RENDER_OVERLAY = False` above to skip the separate map GIF while still running its drawing definitions. This cell writes a separate `_map_outlet_…gif` file.

**Minimum execution order:** run the first four code cells (selection, setup, loading, timing calculation), then this combined GIF cell. The optional preview, drawing-helper, appearance, and standalone GIF cells can be skipped.


In [13]:
import importlib
from rapid_tools import cunge_wave
importlib.reload(cunge_wave)
from rapid_tools.cunge_wave import wave_centroid

from rapid_tools.animation_notebook import prepare_animation
prepare_animation(globals())

# Run setup, data-loading and timing-calculation cells above; no other drawing/export cells are required.
RENDER_COMBINED = False
HYDROGRAPH_VIEW = 'movie'  # 'movie' = playback crop; 'response' = full reviewed window
COMBINED_DPI = 110

MOTION_MODES_TO_SAVE = ['hybrid_midpoint_timing']
WAVE_STYLES_TO_SAVE = ['dot']  # single markers; use hydro_pulse_dots for shoulders

def combined_outlet_series(state):
    hydro = state.outlet_hydro
    seconds = (pd.DatetimeIndex(hydro.time_utc) - origin).total_seconds().to_numpy()
    discharge = hydro.q_outlet_cms.to_numpy(dtype=float)
    if not np.isfinite(discharge).all() or not np.all(np.diff(seconds) > 0):
        raise ValueError('Outlet hydrograph must have finite discharge and increasing UTC timestamps.')
    return seconds, discharge


base_outlet_seconds, base_outlet_q = combined_outlet_series(reference)
variant_outlet_seconds, variant_outlet_q = combined_outlet_series(target)
# The outlet centroid uses exactly the reviewed response baseline and window.
base_centroid, _ = wave_centroid(base_outlet_seconds, base_outlet_q,
    ANIMATION_START_SECONDS, ANIMATION_END_SECONDS, reference.run_row.centroid_baseline_cms)
variant_centroid, _ = wave_centroid(variant_outlet_seconds, variant_outlet_q,
    ANIMATION_START_SECONDS, ANIMATION_END_SECONDS, target.run_row.centroid_baseline_cms)
base_centroid_q = float(np.interp(base_centroid, base_outlet_seconds, base_outlet_q))
variant_centroid_q = float(np.interp(variant_centroid, variant_outlet_seconds, variant_outlet_q))
print(f'Outlet centroid shift (variant − base): {(variant_centroid-base_centroid)/3600:+.3f} h')
print(f'Discharge at each centroid: base {base_centroid_q:.2f}, variant {variant_centroid_q:.2f} m³/s; '
      f'difference {variant_centroid_q-base_centroid_q:+.2f} m³/s')
combined_gifs = []
if RENDER_COMBINED:
    if HYDROGRAPH_VIEW not in {'movie', 'response'}:
        raise ValueError("HYDROGRAPH_VIEW must be 'movie' or 'response'.")
    for MOTION_MODE in MOTION_MODES_TO_SAVE:
        base_movers = build_movers(base_peaks, MOTION_MODE)
        variant_movers = build_movers(variant_peaks, MOTION_MODE)
        frame_times, frame_duration_ms = movie_playback(MOTION_MODE)
        view_start, view_end = ((frame_times[0], frame_times[-1]) if HYDROGRAPH_VIEW == 'movie'
                               else (ANIMATION_START_SECONDS, ANIMATION_END_SECONDS))
        for seconds in [base_outlet_seconds, variant_outlet_seconds]:
            if view_start < seconds[0] or view_end > seconds[-1]:
                raise ValueError('Requested movie/hydrograph window lies outside the saved outlet data.')
        for WAVE_STYLE in WAVE_STYLES_TO_SAVE:
            combined_gif = EXPERIMENT / f'{TARGET_STATE}_vs_{REFERENCE_STATE}_map_outlet_{TIMING_FEATURE}_{MOTION_MODE}_{WAVE_STYLE}.gif'
            # Fixed figure and axes positions keep the map and cursor stable between frames.
            fig, (map_ax, hydro_ax) = plt.subplots(
                1, 2, figsize=(12, 7), gridspec_kw={'width_ratios': [1, 1.35]})
            fig.subplots_adjust(left=0.035, right=0.98, bottom=0.13, top=0.86, wspace=0.20)
            for seconds, discharge, label, color in [
                (base_outlet_seconds, base_outlet_q, f'Base ({REFERENCE_STATE})', '#d62728'),
                (variant_outlet_seconds, variant_outlet_q, f'Variant ({TARGET_STATE})', '#1976d2'),
            ]:
                # Include interpolated view endpoints so the visible curves reach the axes edges.
                inside = (seconds > view_start) & (seconds < view_end)
                plot_seconds = np.r_[view_start, seconds[inside], view_end]
                plot_q = np.interp(plot_seconds, seconds, discharge)
                hydro_ax.plot((plot_seconds - ANIMATION_START_SECONDS) / 3600, plot_q,
                              color=color, linewidth=1.8, label=label)
            hydro_ax.set(xlim=((view_start - ANIMATION_START_SECONDS) / 3600,
                               (view_end - ANIMATION_START_SECONDS) / 3600),
                         xlabel='Hours from reviewed response-window start',
                         ylabel='Outlet discharge (m³/s)', title='Outlet hydrographs')
            for centroid, q, color, label in [
                (base_centroid, base_centroid_q, '#d62728', 'Base centroid'),
                (variant_centroid, variant_centroid_q, '#1976d2', 'Variant centroid')]:
                h = (centroid - ANIMATION_START_SECONDS) / 3600
                hydro_ax.axvline(h, color=color, linestyle=':', linewidth=1.2)
                hydro_ax.plot([h], [q], marker='D', color=color, markersize=7, label=label)
            hydro_ax.grid(alpha=0.20)
            hydro_ax.legend(loc='best', fontsize=9)
            hydro_ax.set_autoscale_on(False)
            cursor = hydro_ax.axvline(0, color='#37474f', linewidth=1.2, linestyle='--')
            base_marker, = hydro_ax.plot([], [], 'o', color='#d62728', markersize=7)
            variant_marker, = hydro_ax.plot([], [], '^', color='#1976d2', markersize=8)
            title = fig.suptitle('', fontsize=12)
            frames = []
            try:
                for current_time in frame_times:
                    map_ax.clear()
                    draw_overlay(map_ax, current_time)
                    map_ax.set_title(f'Network {TIMING_FEATURE} positions')
                    hours = (current_time - ANIMATION_START_SECONDS) / 3600
                    cursor.set_xdata([hours, hours])
                    base_marker.set_data([hours], [np.interp(current_time, base_outlet_seconds, base_outlet_q)])
                    variant_marker.set_data([hours], [np.interp(current_time, variant_outlet_seconds, variant_outlet_q)])
                    title.set_text(f'Network {network} · {TARGET_STATE} vs {REFERENCE_STATE}\n'
                                   f'{TIMING_FEATURE} / {MOTION_MODE} / {WAVE_STYLE} · window +{hours:.2f} h')
                    # Use a constant pixel size instead of tight bounding boxes per frame.
                    fig.set_dpi(COMBINED_DPI)
                    fig.canvas.draw()
                    rgba = np.asarray(fig.canvas.buffer_rgba())
                    frames.append(Image.fromarray(rgba).convert('RGB'))
                combined_gif.parent.mkdir(parents=True, exist_ok=True)
                frames[0].save(combined_gif, save_all=True, append_images=frames[1:],
                               duration=frame_duration_ms, loop=0, disposal=2, optimize=False)
            finally:
                plt.close(fig)
                del frames
            combined_gifs.append(combined_gif)
            print(f'Wrote {combined_gif.name}: {len(frame_times)} frames, '
                  f'{sum(frame_duration_ms) / 1000:.2f} seconds; map and outlet cursor share one clock.')
else:
    print('Combined GIF skipped (RENDER_COMBINED=False).')


Outlet centroid shift (variant − base): -7.461 h
Discharge at each centroid: base 2503.14, variant 2461.51 m³/s; difference -41.62 m³/s
Movie: window +0.00 to +134.30 h; 500 frames, 10.00 s playback (frame delays 20–20 ms)
Wrote CB001_bubble_3_vs_S000_base_map_outlet_centroid_hybrid_midpoint_timing_dot.gif: 500 frames, 10.00 seconds; map and outlet cursor share one clock.


## Flood pulse: separate networks and shared outlet hydrographs

Both complete networks are drawn separately with the same extent. Each **subreach** brightens and thickens according to its own discharge at the shared animation timestamp. Outlet curves retain the original modelled discharge.

The default **local pulse display** subtracts a straight background line between each subreach's own start/end discharge in the full reviewed window, then displays only positive departures. This returns every reach to grey at both window endpoints, even if final discharge differs from initial discharge. It is a visual background choice, not physical baseflow separation or a replacement for the reviewed metric baseline.

`PULSE_HIGHLIGHT_LEVEL = 0.80` emphasizes the upper 20% of each local pulse; everything below this threshold stays grey. Lower it towards 0 to show broader rise/recession. `PULSE_BACKGROUND = 'reviewed'` restores the fixed-baseline view. Normalization always uses the full reviewed window, not the movie crop. A broad pulse can still occupy many reaches at the same time; this display does not force a moving narrow packet.

Run the first four code cells, then this cell. Existing movie start/end and playback settings apply. Previous marker exports are disabled while trying this view.

Both maps share a fixed **viridis 0–100% scale** (purple → green → yellow). The cutoff only hides low intensity; it does not rescale the colours. Try `PULSE_HIGHLIGHT_LEVEL = 0.30` to see more of the rise and recession. Dashed outlet guides show the selected fraction above each outlet background, and shading identifies discharge above that guide. These are local relative intensities, not equal absolute discharge across reaches or states.


In [25]:
# Actual evolving discharge: no peak/centroid trajectory or artificial time shifts.
RENDER_FLOOD_PULSE = True
PULSE_N_FRAMES = 240
PULSE_DPI = 100
PULSE_BACKGROUND = 'local_endpoints'  # local start-to-end line; 'reviewed' restores fixed baseline
PULSE_HIGHLIGHT_LEVEL = 0.0  # try 0.30 for broader rise/fall; below cutoff stays grey
PULSE_CMAP = 'viridis'  # shared fixed 0–100% scale; yellow marks the local crest

import importlib
from rapid_tools import flood_pulse
importlib.reload(flood_pulse)
from rapid_tools import animation_notebook
importlib.reload(animation_notebook)
animation_notebook.prepare_animation(globals())
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

if not 0 <= PULSE_HIGHLIGHT_LEVEL < 1:
    raise ValueError('PULSE_HIGHLIGHT_LEVEL must be >=0 and <1.')

if RENDER_FLOOD_PULSE:
    pulse_times, pulse_delays = movie_playback(MOTION_MODE, frame_count=PULSE_N_FRAMES)
    pulse_start, pulse_end = pulse_times[0], pulse_times[-1]
    pulse_fields = [flood_pulse.prepare_pulse_field(
        state.wave_stack.reaches, discharge, time_seconds,
        ANIMATION_START_SECONDS, ANIMATION_END_SECONDS, state.run_row.centroid_baseline_cms, background=PULSE_BACKGROUND)
        for state, discharge in [(reference, reference_qout), (target, target_qout)]]
    pulse_fig = plt.figure(figsize=(15, 6.5), dpi=PULSE_DPI)
    grid = pulse_fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.7],
                                left=0.025, right=0.98, top=0.83, bottom=0.17, wspace=0.25)
    map_axes = [pulse_fig.add_subplot(grid[0,0]), pulse_fig.add_subplot(grid[0,1])]
    hydro_ax = pulse_fig.add_subplot(grid[0,2])
    layers=[]
    pulse_cmap = plt.get_cmap(PULSE_CMAP).copy()
    pulse_cmap.set_bad('#dce1e5')
    pulse_norm = Normalize(0, 1, clip=True)
    for ax, field, label in zip(map_axes, pulse_fields,
            ['Full complexity', 'Simplified']):
        layer = LineCollection(field['paths'], cmap=pulse_cmap, norm=pulse_norm,
                               linewidths=0.7, capstyle='round')
        ax.add_collection(layer);layers.append(layer)
        ax.set(xlim=(all_bounds[0],all_bounds[2]), ylim=(all_bounds[1],all_bounds[3]),
               aspect='equal', title=label)
        ax.set_axis_off();draw_scale_bar(ax)
    colorbar = pulse_fig.colorbar(ScalarMappable(norm=pulse_norm,cmap=pulse_cmap),
                                 ax=map_axes, orientation='horizontal', fraction=0.035, pad=0.035)
    colorbar.set_ticks([0,.25,.5,.75,1], labels=['0%','25%','50%','75%','100%'])
    colorbar.ax.axvline(PULSE_HIGHLIGHT_LEVEL, color='black', linestyle='--', linewidth=1)
    colorbar.set_label(f'Local pulse intensity',fontsize=12)
    outlet_series=[]
    for state,color,label in [(reference,'#d62728','Full complexity'),(target,'#1976d2','Simplified')]:
        hydro=state.outlet_hydro
        t=(pd.DatetimeIndex(hydro.time_utc)-origin).total_seconds().to_numpy()
        q=hydro.q_outlet_cms.to_numpy(float)
        if not t[0] <= pulse_start < pulse_end <= t[-1]:
            raise ValueError('Outlet hydrograph')
        view_t=np.r_[pulse_start,t[(t>pulse_start)&(t<pulse_end)],pulse_end]
        hydro_ax.plot((view_t-ANIMATION_START_SECONDS)/3600,np.interp(view_t,t,q),
                      color=color,label=label,linewidth=1.8)
        # Normalize the outlet pulse on the FULL reviewed window, just as on the maps.
        # Use the original outlet samples, including interpolated window boundaries.
        window_t = np.r_[ANIMATION_START_SECONDS,
                         t[(t>ANIMATION_START_SECONDS)&(t<ANIMATION_END_SECONDS)],
                         ANIMATION_END_SECONDS]
        window_q = np.interp(window_t,t,q)
        if PULSE_BACKGROUND == 'local_endpoints':
            window_background = np.interp(window_t, window_t[[0,-1]], window_q[[0,-1]])
            view_background = np.interp(view_t, window_t[[0,-1]], window_q[[0,-1]])
        else:
            outlet_baseline = min(float(state.run_row.centroid_baseline_cms), window_q.min())
            window_background = np.full_like(window_t,outlet_baseline)
            view_background = np.full_like(view_t,outlet_baseline)
        outlet_scale = max(0., float(np.max(window_q-window_background)))
        threshold_q = view_background + PULSE_HIGHLIGHT_LEVEL*outlet_scale
        view_hours = (view_t-ANIMATION_START_SECONDS)/3600
        view_q = np.interp(view_t,t,q)
        # hydro_ax.plot(view_hours,threshold_q, color='gray', linestyle='--', linewidth=1,
        #               label=f'{label}: {PULSE_HIGHLIGHT_LEVEL:.0%} pulse')
        # hydro_ax.fill_between(view_hours,threshold_q,view_q,
        #                       where=(view_q>=threshold_q)&(outlet_scale>0),
        #                       interpolate=True,color=color,alpha=0.16)
        outlet_series.append((t,q))
    hydro_ax.set(xlim=((pulse_start-ANIMATION_START_SECONDS)/3600,(pulse_end-ANIMATION_START_SECONDS)/3600),
                 xlabel='Hours',ylabel='Discharge (m³/s)',
                 title='Outlet response')
    hydro_ax.grid(alpha=0.2);hydro_ax.legend(loc='upper right');hydro_ax.set_autoscale_on(False)
    pulse_cursor=hydro_ax.axvline(0,color='black',linestyle='--',linewidth=1.2)
    pulse_markers=[hydro_ax.plot([],[],marker='o',color=color,markersize=6)[0]
                   for color in ['#d62728','#1976d2']]
    pulse_title=pulse_fig.suptitle('',fontsize=13)
    # pulse_fig.text(0.5,0.025,
    #     f'Maps: highlight above {PULSE_HIGHLIGHT_LEVEL:.0%} of each local pulse · background: {PULSE_BACKGROUND}. '
    #     'Hydrographs: original modelled discharge.',ha='center',fontsize=9)

    def update_flood_pulse(current_time):
        for layer,field in zip(layers,pulse_fields):
            intensity=flood_pulse.pulse_intensity(field,current_time)
            highlight=np.clip((intensity-PULSE_HIGHLIGHT_LEVEL)/(1-PULSE_HIGHLIGHT_LEVEL),0,1)
            layer.set_array(np.ma.masked_where(intensity<PULSE_HIGHLIGHT_LEVEL,intensity))
            layer.set_linewidths(0.6+2.4*highlight)
        hours=(current_time-ANIMATION_START_SECONDS)/3600
        pulse_cursor.set_xdata([hours,hours])
        for marker,(t,q) in zip(pulse_markers,outlet_series):
            marker.set_data([hours],[np.interp(current_time,t,q)])
        pulse_title.set_text(f'Susitna river example\n'
                             f'Full complexity vs Simplified\n'
                             f'Hours {hours:.2f}')

    pulse_gif = EXPERIMENT / f'{TARGET_STATE}_vs_{REFERENCE_STATE}_flood_pulse_outlets_{PULSE_BACKGROUND}_{PULSE_HIGHLIGHT_LEVEL:.0%}.gif'
    pulse_frames=[]
    try:
        for current_time in pulse_times:
            update_flood_pulse(current_time)
            pulse_fig.canvas.draw()
            pulse_frames.append(Image.fromarray(np.asarray(pulse_fig.canvas.buffer_rgba())).convert('RGB'))
        pulse_gif.parent.mkdir(parents=True,exist_ok=True)
        pulse_frames[0].save(pulse_gif,save_all=True,append_images=pulse_frames[1:],
                            duration=pulse_delays,loop=0,disposal=2,optimize=False)
    finally:
        plt.close(pulse_fig)
        del pulse_frames
    print(f'Wrote {pulse_gif} ({len(pulse_times)} frames, {sum(pulse_delays)/1000:.2f} seconds)')
    print('Both maps show their complete networks independently. Baselines lowered locally where needed:',
          [field['baseline_lowered_count'] for field in pulse_fields])
else:
    print('Flood-pulse GIF skipped (RENDER_FLOOD_PULSE=False).')


Movie: window +0.00 to +134.30 h; 240 frames, 20.00 s playback (frame delays 80–90 ms)
Wrote /Users/6256481/Code/river-hierarchy/network_variants/output/37_compound_bubbles__anchored/CB001_bubble_3_vs_S000_base_flood_pulse_outlets_local_endpoints_0%.gif (240 frames, 20.00 seconds)
Both maps show their complete networks independently. Baselines lowered locally where needed: [0, 0]


## Reading the result

Perfect pre-collapse agreement is a hypothesis to test, not a guaranteed property: it requires identical forcing, a consistent event window, identical parent-link aggregation, and matching peak rules. If dots differ upstream of the collapse, inspect the `matched` table first; common causes are different subreach discretisation, an ambiguous peak plateau, or a changed event-window alignment.